# 04 — Trois algorithmes, une architecture lue, une confiance interprétée

**Projet** : Extraction d'entités nommées par spaCy — le tagger appris, corroboré par des règles

## Objectifs pédagogiques

1. Comparer les trois algorithmes de la stack sur la **validation** : règles, tagger, hybride.
1. Lire l'architecture spaCy réellement construite (le projet ne la redéclare pas, il la publie).
1. Comprendre ce que « confiance » veut dire ici : un niveau de **corroboration** par les règles, pas une probabilité.
1. Arbitrer un seuil d'automatisation sur le split de calibration, puis le lire sur la validation.

## 1. Mise en place

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.data.schemas import (  # noqa: E402
    ENTITY_LABELS,
)
from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un corpus réduit (320 messages) et un budget d'époques
# réduit : l'exécution reste rapide, et les valeurs absolues d'un notebook ne sont pas celles de la
# référence publiée par le README.
NB_DOCUMENTS = 320
NB_EPOCHS = 3

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                f"train.epochs={NB_EPOCHS}",
                "seed=42",
                "log_level=WARNING",
                "train.callbacks.progress_bar=false",
                "train.callbacks.logging_every=100",
            ],
        )
    )

MODEL_NODE = dict(node(CONFIG, "model"))
TEXT_COLUMN = str(MODEL_NODE.get("text_column", "text"))
TARGET_COLUMN = str(MODEL_NODE.get("target", "label"))
ID_COLUMN = str(CONFIG.data.id_column or "msg_id")
SPLIT_COLUMN = str(CONFIG.data.group_column or "split")

# Les notebooks lisent **et écrivent** dans leur propre bac à sable : le corpus réduit et les
# artefacts des notebooks ne touchent ni `data/raw` ni `artifacts/`, donc ceux de `make train`
# restent ceux de la référence.
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=PROJECT_ROOT / "outputs" / "notebooks" / "data",
    artifacts_dir=PROJECT_ROOT / "outputs" / "notebooks" / "artifacts",
).ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil contractuel : 0.7)")
print(f"Algorithme servi  : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Types d'entités   : {len(ENTITY_LABELS)} — {', '.join(ENTITY_LABELS)}")
print(f"Bac à sable       : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

In [ ]:
from src.data.generators import SyntheticEntityCorpusGenerator
from src.data.loaders import EntityCorpusLoader

LOADER = EntityCorpusLoader(
    NB_PATHS,
    dataset_name=str(CONFIG.data.dataset_name),
    formats=tuple(CONFIG.data.formats),
    validation_enabled=bool(CONFIG.data.validation.raw),
    lazy_validation=bool(CONFIG.data.validation.lazy),
)

if LOADER.documents_path.exists():
    DOCUMENTS, SPANS = LOADER.load_corpus()
    METADATA = LOADER.load_metadata()
    print(f"Corpus lu depuis le bac à sable : {len(DOCUMENTS)} messages, graine {METADATA['seed']}")
else:
    # Un clone frais n'a pas encore de données : le notebook reste exécutable. Le corpus réduit est
    # persisté **dans le bac à sable** pour que les notebooks suivants et les pipelines travaillent
    # sur exactement les mêmes fichiers.
    bundle = SyntheticEntityCorpusGenerator.from_config(CONFIG.data).generate()
    DOCUMENTS, SPANS, METADATA = bundle.documents, bundle.queries, dict(bundle.metadata)
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_annotations(SPANS)
    LOADER.save_metadata(METADATA)
    print(f"Bac à sable vide : corpus de {len(DOCUMENTS)} messages généré et persisté")

DOCUMENTS = DOCUMENTS.sort_values(ID_COLUMN).reset_index(drop=True)
SPANS = SPANS.sort_values([ID_COLUMN, "start", "end"]).reset_index(drop=True)

TRAIN_DOCUMENTS = LOADER.split("train", frame=DOCUMENTS)
VAL_DOCUMENTS = LOADER.split("val", frame=DOCUMENTS)
CALIBRATION_DOCUMENTS = LOADER.split("calibration", frame=DOCUMENTS)
TEST_DOCUMENTS = LOADER.split("test", frame=DOCUMENTS)
TRAIN_SPANS = LOADER.split_annotations("train", documents=DOCUMENTS, spans=SPANS)
VAL_SPANS = LOADER.split_annotations("val", documents=DOCUMENTS, spans=SPANS)
TEST_SPANS = LOADER.split_annotations("test", documents=DOCUMENTS, spans=SPANS)

print(
    "messages par split :",
    {
        name: len(frame)
        for name, frame in (
            ("train", TRAIN_DOCUMENTS),
            ("val", VAL_DOCUMENTS),
            ("calibration", CALIBRATION_DOCUMENTS),
            ("test", TEST_DOCUMENTS),
        )
    },
)
print(
    "mentions par split :",
    {
        name: len(frame)
        for name, frame in (("train", TRAIN_SPANS), ("val", VAL_SPANS), ("test", TEST_SPANS))
    },
)

## 1. Les algorithmes que la stack sait servir

La fabrique est la seule porte d'entrée : le reste du projet ne connaît que le contrat `BaseEntityTagger`. Les trois algorithmes partagent donc exactement les mêmes métriques et le même format de sortie.

In [ ]:
from src.models import ALGORITHMS, available_algorithms, build_model, describe_algorithm

SERVED_ALGORITHM = str(CONFIG.model.algorithm)
ALGORITHMS_TABLE = pd.DataFrame(
    [describe_algorithm(name) for name in available_algorithms(CONFIG.metrics.task)]
).set_index("name")
display(ALGORITHMS_TABLE[["display_name", "learns_weights", "rationale"]])
print("algorithmes déclarés :", list(ALGORITHMS), "| algorithme servi :", SERVED_ALGORITHM)
print("défauts du registre :", describe_algorithm(SERVED_ALGORITHM)["defaults"])
print("hyper-paramètres du manifeste :", dict(MODEL_NODE.get("params", {}) or {}))

**Ce qu'il faut retenir**

- `learns_weights` sépare deux familles : un gazetteer **ne généralise pas** (il reconnaît), un tagger apprend des représentations.
- Les défauts viennent du registre, les hyper-paramètres du manifeste : comparer deux algorithmes à paramètres explicites (`{}`) est la seule comparaison loyale.
- Le contrat commun est ce qui permet de changer d'algorithme sans toucher ni au trainer, ni à l'évaluateur, ni à l'inférence.

## 2. Comparaison sur la validation

Budget réduit (`__NB_EPOCHS__` époques) et corpus réduit : ces chiffres servent à **choisir**, pas à publier. Un tagger neural à budget réduit n'a pas fini d'apprendre — le fait est visible dans le tableau, et c'est en soi une information.

In [ ]:
from src.evaluation.evaluator import EntityEvaluator


def evaluate_on_validation(model, name: str) -> dict[str, float]:
    """Fit one algorithm on the training split and score the validation split."""
    model.fit(TRAIN_DOCUMENTS, TRAIN_SPANS)
    result = EntityEvaluator(
        model,
        config=CONFIG.model_dump(),
        paths=NB_PATHS,
        primary_metric=CONFIG.metrics.primary,
        min_primary_metric=CONFIG.metrics.min_primary,
    ).evaluate(VAL_DOCUMENTS, VAL_SPANS)
    return {
        "algorithme": name,
        "poids appris": describe_algorithm(name)["learns_weights"],
        "entity_f1 (micro)": result.metrics["entity_f1"],
        "macro_f1": result.metrics["macro_f1"],
        "precision": result.metrics["entity_precision"],
        "rappel": result.metrics["entity_recall"],
        "bornes exactes": result.metrics["boundary_accuracy"],
        "p50 (ms)": result.metrics["latency_p50_ms"],
        "mentions prédites": len(result.predictions),
    }


COMPARISON = pd.DataFrame(
    [
        evaluate_on_validation(build_model(CONFIG, algorithm=name, params={"epochs": 3}), name)
        for name in available_algorithms(CONFIG.metrics.task)
    ]
).set_index("algorithme")
display(COMPARISON.round(4))
print(
    "meilleur micro-F1 :",
    COMPARISON["entity_f1 (micro)"].idxmax(),
    "| meilleur macro-F1 :",
    COMPARISON["macro_f1"].idxmax(),
)

**Ce qu'il faut retenir**

- Le **micro**-F1 récompense les types fréquents, le **macro**-F1 compte chaque type à égalité : un algorithme qui sacrifie le transporteur gagne au micro et perd au macro.
- Les règles seules obtiennent un score qui ne doit pas surprendre : sur les types à motif, elles sont excellentes — c'est mesuré au notebook 01.
- Un écart de score plus petit que la variabilité de la graine n'est pas un gain ; le notebook 05 montre comment mesurer cette variabilité.

## 3. L'architecture réellement construite

Le projet ne **redéclare** pas l'architecture du réseau : il la lit sur le pipeline servi et la publie dans la fiche modèle. Une configuration partielle produirait un modèle différent de celui que la fiche décrit.

In [ ]:
from dataclasses import asdict

SERVED = build_model(CONFIG)  # algorithme et hyper-paramètres du manifeste
SERVED.fit(TRAIN_DOCUMENTS, TRAIN_SPANS)
ARCHITECTURE = SERVED.architecture_report()
print("pipes spaCy      :", ARCHITECTURE["pipe_names"])
print("architecture NER :", ARCHITECTURE.get("architecture"))
print("tok2vec          :", ARCHITECTURE.get("tok2vec"))
print()
CARD = SERVED.model_card(artifact="tagger")
display(
    pd.Series(
        {key: str(value) for key, value in asdict(CARD).items() if key != "feature_names"}
    ).to_frame("fiche de modèle")
)
print("types servis par le modèle :", SERVED.labels)

**Ce qu'il faut retenir**

- `architecture_report()` lit le pipeline : HashEmbedCNN v2 (largeur 96, fenêtre 1, `subword` actif) et un tok2vec partagé — l'équivalent explicite du défaut du pipe `ner`.
- La fiche modèle contient les **versions** des bibliothèques : un artefact retrouvé six mois plus tard reste interprétable.
- Les types servis sont appris du corpus (`labels`), pas recopiés d'une configuration : un type absent du train n'existe pas pour le modèle.

## 4. La confiance est une corroboration, pas une probabilité

La stack publie, à côté de chaque mention, une provenance (`regle` ou `modele`) et une confiance : la part du span confirmée par la couche de règles. Une confiance de 1,0 signifie « corroborée par une règle », pas « certaine ».

In [ ]:
from src.inference.predictor import EntityPredictor
from src.training.metrics import confidence_gap, confidence_table

PREDICTOR = EntityPredictor(SERVED, config=CONFIG.model_dump(), id_column=ID_COLUMN)
PREDICTED_VAL = PREDICTOR.predict(VAL_DOCUMENTS, spans=VAL_SPANS)
display(confidence_table(PREDICTED_VAL, VAL_SPANS, bins=5))
print("provenances :", PREDICTED_VAL["source"].value_counts().to_dict())
print(
    "confiance moyenne par provenance :",
    PREDICTED_VAL.groupby("source")["confidence"].mean().round(3).to_dict(),
)
print(
    "exactitude par provenance :",
    PREDICTED_VAL.groupby("source")["correct"].mean().round(3).to_dict(),
)
print(
    "écart |confiance moyenne des bonnes mentions - 1| :", confidence_gap(PREDICTED_VAL, VAL_SPANS)
)

**Ce qu'il faut retenir**

- Les tranches hautes de confiance sont plus précises que les basses — sinon la colonne ne servirait à rien, et le projet le dirait.
- `confidence_gap` mesure la distance entre la confiance annoncée et la précision observée : un écart élevé interdit d'appeler cette colonne une probabilité.
- Les mentions `regle` sont exactes par construction sur les surfaces apprises : c'est la partie du score qui vient de la liste, pas du modèle.

## 5. Un seuil d'automatisation, arbitré sur la calibration

Si un conseiller ne relit que les mentions douteuses, il faut choisir le seuil : au-dessus, la mention est validée automatiquement. Le seuil se règle sur le split de **calibration** et se lit ensuite sur la validation — jamais sur le test.

In [ ]:
CALIBRATION_SPANS = LOADER.split_annotations("calibration", documents=DOCUMENTS, spans=SPANS)
CALIBRATED = PREDICTOR.predict(CALIBRATION_DOCUMENTS, spans=CALIBRATION_SPANS)

rows = []
for threshold in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9]:
    selected = CALIBRATED[CALIBRATED["confidence"] >= threshold]
    rows.append(
        {
            "seuil": threshold,
            "mentions validées": len(selected),
            "couverture": round(len(selected) / max(len(CALIBRATED), 1), 4),
            "précision observée": round(float(selected["correct"].mean()), 4)
            if len(selected)
            else None,
        }
    )
GRID = pd.DataFrame(rows)
display(GRID)
eligible = GRID[GRID["précision observée"].notna()]
tenable = eligible[eligible["précision observée"] >= 0.95]
if len(tenable):
    chosen = float(tenable["seuil"].min())
    print(f"seuil retenu : {chosen} (précision visée {0.95})")
else:
    chosen = float(eligible["seuil"].max())
    print(f"aucun seuil n'atteint {0.95} sur la calibration : seuil le plus haut retenu ({chosen})")

In [ ]:
AUTOMATED = PREDICTED_VAL[PREDICTED_VAL["confidence"] >= chosen]
print(f"seuil {chosen} appliqué à la validation :")
print(
    f"  mentions auto-validées : {len(AUTOMATED)}/{len(PREDICTED_VAL)} "
    f"({len(AUTOMATED) / max(len(PREDICTED_VAL), 1):.1%})"
)
print(
    f"  précision observée     : {float(AUTOMATED['correct'].mean()):.4f}"
    if len(AUTOMATED)
    else "  aucune mention au-dessus du seuil"
)
print(f"  mentions à relire      : {len(PREDICTED_VAL) - len(AUTOMATED)}")

**Ce qu'il faut retenir**

- Le seuil est un **arbitrage de service** : la couverture dit combien de mentions on cesse de relire, la précision dit ce que cela coûte.
- Régler le seuil sur la calibration et le lire sur la validation est la même discipline que pour un modèle : le test reste vierge jusqu'au notebook 06.
- Un seuil sans mesure de précision n'est pas un seuil, c'est une espérance.

## Synthèse

| Algorithme | Ce qu'il apprend | Force | Limite mesurée |
| --- | --- | --- | --- |
| `gazetteer` | rien (motifs + index du train) | types à motif, latence minimale | surfaces réservées introuvables |
| `tagger` | représentations de tokens | généralise aux formes inédites | budget d'entraînement, types minoritaires |
| `hybride` | les deux | score des motifs + généralisation | complexité de lecture |

**Suite** : `05_training.ipynb` exécute le pipeline réel, dans un bac à sable.